In [12]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter

import librosa
import soundfile as sf
import joblib
import pretty_midi

from sklearn.ensemble import RandomForestClassifier


In [13]:
PROJECT_PATH = os.getcwd()

AUDIO_DIR = os.path.join(PROJECT_PATH, "data", "audio")
CSV_PATH = os.path.join(PROJECT_PATH, "data", "annotations", "raga_labels.csv")

MODEL_DIR = os.path.join(PROJECT_PATH, "outputs", "models")
MIDI_DIR = os.path.join(PROJECT_PATH, "outputs", "midi")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(MIDI_DIR, exist_ok=True)

print("Setup complete.")


Setup complete.


In [14]:
def ensure_wav(filename):
    name, ext = os.path.splitext(filename)
    ext = ext.lower()

    wav_path = os.path.join(AUDIO_DIR, name + ".wav")
    src_path = os.path.join(AUDIO_DIR, filename)

    if ext == ".wav" and os.path.exists(wav_path):
        return wav_path

    if ext == ".mp3":
        y, sr = librosa.load(src_path, sr=None)
        sf.write(wav_path, y, sr)
        return wav_path

    if os.path.exists(wav_path):
        return wav_path

    return None


In [15]:
def extract_features(filename, duration=45.0):
    wav_path = ensure_wav(filename)
    if wav_path is None:
        return None, None, None

    y, sr = librosa.load(wav_path, sr=22050)

    # Trim silence
    y, _ = librosa.effects.trim(y, top_db=30)
    y = y[: int(duration * sr)]

    if len(y) < sr * 5:
        return None, None, None

    # Pitch extraction
    f0, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("A1"),
        fmax=librosa.note_to_hz("C8")
    )

    valid = ~np.isnan(f0)
    if not np.any(valid):
        return None, None, None

    midi = librosa.hz_to_midi(f0[valid])
    pcs = np.mod(midi, 12)

    # Tonic estimation
    tonic = np.argmax(np.bincount(pcs.astype(int), minlength=12))

    # Relative pitch histogram
    rel = np.mod(pcs - tonic, 12)
    hist = np.bincount(rel.astype(int), minlength=12)
    hist = hist / np.sum(hist)

    return hist, tonic, pcs


In [16]:
df = pd.read_csv(CSV_PATH)

X, y_labels = [], []

print("Extracting features...")

for _, row in tqdm(df.iterrows(), total=len(df)):
    feat, tonic, _ = extract_features(row["filename"])
    if feat is None:
        continue
    X.append(feat)
    y_labels.append(row["raga"])

X = np.array(X)
y_labels = np.array(y_labels)

print("Samples:", len(X))
print("Ragas:", Counter(y_labels))


Extracting features...


100%|██████████| 56/56 [13:18<00:00, 14.25s/it]

Samples: 55
Ragas: Counter({'Abhogi': 1, 'Ahir Bhairav': 1, 'Bageshree': 1, 'Bahar': 1, 'Bairagi': 1, 'Bhairav': 1, 'Bhairavi': 1, 'Bhatiyar': 1, 'Bhimpalasi': 1, 'Bhupali': 1, 'Bibhas': 1, 'Bihag': 1, 'Bilaskhani Todi': 1, 'Brindawani Malhar': 1, 'Brindawani Saarang': 1, 'Darbari Kanada': 1, 'Desh': 1, 'Dhani': 1, 'Durga': 1, 'Ganapati': 1, 'Gaud Malhar': 1, 'Hameer': 1, 'Hamsadhwani': 1, 'Jog': 1, 'Kalyan': 1, 'Kedar': 1, 'Khamaj': 1, 'Kirwani': 1, 'Komal Bhimpalasi': 1, 'Lagan Gandhar': 1, 'Lalit Pancham': 1, 'Lalit': 1, 'Madhukauns': 1, 'Malkauns': 1, 'Marwa': 1, 'Megh': 1, 'Miyan Malhar': 1, 'Multani': 1, 'Nata Bhairavi': 1, 'Patadeep': 1, 'Puriya Dhanashree': 1, 'Puriya': 1, 'Rageshri': 1, 'Ramdasi Malhar': 1, 'Sarang Malhar': 1, 'Saraswathi': 1, 'Sawani': 1, 'Shree': 1, 'Shuddha Bilawal': 1, 'Shuddha Sarang': 1, 'Sindhu Bhairavi': 1, 'Tilak Kamod': 1, 'Todi': 1, 'Virat Bhairav': 1, 'Yaman': 1})


In [18]:
clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

clf.fit(X, y_labels)

MODEL_PATH = os.path.join(MODEL_DIR, "raga_random_forest.joblib")

joblib.dump(
    {
        "model": clf,
        "feature_type": "relative_pitch_histogram"
    },
    MODEL_PATH
)

print("Model trained & saved.")


Model trained & saved.


In [19]:
bundle = joblib.load(MODEL_PATH)
clf = bundle["model"]

print("Model loaded successfully.")


Model loaded successfully.


In [20]:
TEST_AUDIO = "Yaman.wav"  # or MP3

feat, test_tonic, melody_pcs = extract_features(TEST_AUDIO)

if feat is None:
    raise ValueError("Feature extraction failed. Use a clean melodic audio.")

predicted_raga = clf.predict(feat.reshape(1, -1))[0]

print("🎵 Predicted Raga:", predicted_raga)


🎵 Predicted Raga: Yaman


In [21]:
raga_scales = {

    # --- Audava / Pentatonic ---
    "Abhogi": [0, 3, 5, 7, 10],
    "Bhupali": [0, 2, 4, 7, 9],
    "Durga": [0, 2, 4, 7, 9],
    "Dhani": [0, 3, 5, 7, 10],
    "Hamsadhwani": [0, 2, 4, 7, 11],
    "Malkauns": [0, 3, 5, 8, 10],
    "Chandrakauuns": [0, 3, 5, 8, 11],
    "Madhukauns": [0, 3, 5, 8, 11],
    "Megh": [0, 2, 5, 7, 10],

    # --- Bhairav ang ---
    "Bhairav": [0, 1, 4, 5, 7, 8, 11],
    "Ahir Bhairav": [0, 2, 4, 5, 7, 8, 10],
    "Bairagi": [0, 1, 4, 5, 7, 8, 11],
    "Bibhas": [0, 1, 4, 5, 7, 8, 11],
    "Virat Bhairav": [0, 1, 4, 6, 7, 8, 11],

    # --- Bhairavi / Asavari ang ---
    "Bhairavi": [0, 1, 3, 5, 7, 8, 10],
    "Sindhu Bhairavi": [0, 1, 3, 5, 7, 8, 10, 11],
    "Asavari": [0, 2, 3, 5, 7, 8, 10],
    "Jaunpuri": [0, 2, 3, 5, 7, 8, 10],
    "Komal Bhimpalasi": [0, 3, 5, 7, 10],

    # --- Kafi ang ---
    "Bageshree": [0, 3, 5, 7, 10],
    "Bhimpalasi": [0, 3, 5, 7, 10],
    "Darbari Kanada": [0, 2, 3, 5, 7, 9, 10],
    "Jog": [0, 3, 5, 7, 10],
    "Rageshri": [0, 2, 5, 7, 9, 10],

    # --- Kalyan ang ---
    "Yaman": [0, 2, 4, 6, 7, 9, 11],
    "Kalyan": [0, 2, 4, 6, 7, 9, 11],
    "Bihag": [0, 4, 6, 7, 9, 11],
    "Hameer": [0, 2, 4, 6, 7, 9, 11],
    "Tilak Kamod": [0, 2, 4, 7, 9, 11],
    "Kedar": [0, 2, 4, 7, 9, 11],

    # --- Todi / Marwa ang ---
    "Todi": [0, 1, 3, 6, 7, 8, 11],
    "Bilaskhani Todi": [0, 1, 3, 5, 7, 8, 10],
    "Multani": [0, 1, 3, 6, 7, 8, 11],
    "Marwa": [0, 1, 4, 6, 7, 9, 11],
    "Puriya": [0, 1, 4, 6, 7, 8, 11],
    "Puriya Dhanashree": [0, 1, 4, 6, 7, 9, 11],
    "Shree": [0, 1, 4, 6, 7, 8, 11],

    # --- Bilawal / Khamaj ang ---
    "Shuddha Bilawal": [0, 2, 4, 5, 7, 9, 11],
    "Khamaj": [0, 2, 4, 5, 7, 9, 10],
    "Desh": [0, 2, 4, 5, 7, 9, 10],
    "Tilak Kamod": [0, 2, 4, 7, 9, 11],

    # --- Malhar ang ---
    "Bahar": [0, 2, 4, 5, 7, 9, 10],
    "Brindawani Malhar": [0, 2, 5, 7, 9],
    "Gaud Malhar": [0, 2, 4, 5, 7, 9, 10],
    "Miyan Malhar": [0, 2, 4, 5, 7, 9, 10],
    "Ramdasi Malhar": [0, 2, 4, 5, 7, 9, 10],
    "Sarang Malhar": [0, 2, 5, 7, 9],
    "Sawani": [0, 2, 5, 7, 9],

    # --- Sarang / Saarang ---
    "Brindawani Saarang": [0, 2, 5, 7, 9],
    "Shuddha Sarang": [0, 2, 4, 6, 7, 9, 11],
    "Patadeep": [0, 2, 4, 5, 7, 9],

    # --- Misc / Carnatic overlap ---
    "Kirwani": [0, 2, 3, 5, 7, 8, 11],
    "Nata Bhairavi": [0, 2, 3, 5, 7, 8, 10],
    "Saraswathi": [0, 2, 4, 6, 7, 9, 11],
    "Ganapati": [0, 2, 4, 7, 9],
    "Lalit": [0, 1, 4, 6, 7, 8, 11],
    "Lalit Pancham": [0, 1, 4, 6, 7, 8, 11],
    "Lagan Gandhar": [0, 2, 3, 5, 7, 9, 10],
}


In [22]:
def generate_chords(scale):
    chords = []
    for i in range(len(scale)):
        for j in range(i+1, len(scale)):
            for k in range(j+1, len(scale)):
                chords.append([scale[i], scale[j], scale[k]])
    return chords


def select_best_chords(chords, melody_pcs, top_k=4):
    scored = []
    for c in chords:
        score = sum(1 for p in melody_pcs if p in c)
        scored.append((c, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [c for c, _ in scored[:top_k]]


scale = raga_scales[predicted_raga]
all_chords = generate_chords(scale)
selected_chords = select_best_chords(all_chords, melody_pcs)


In [23]:
safe_name = predicted_raga.replace(" ", "_")
OUTPUT_MIDI = os.path.join(MIDI_DIR, f"{safe_name}_harmony.mid")

pm = pretty_midi.PrettyMIDI()
inst = pretty_midi.Instrument(program=0)

base = 60 + test_tonic
t = 0

for chord in selected_chords:
    for pc in chord:
        note = pretty_midi.Note(
            velocity=80,
            pitch=base + pc,
            start=t,
            end=t + 2
        )
        inst.notes.append(note)
    t += 2

pm.instruments.append(inst)
pm.write(OUTPUT_MIDI)

print("🎹 Harmony MIDI saved at:", OUTPUT_MIDI)


🎹 Harmony MIDI saved at: c:\Users\Sahith Raj\Desktop\RAGA_Harmony\outputs\midi\Yaman_harmony.mid
